# 🌾 Segmentação Cadastral de Piquetes de Pastagem - LAPIG / UFG

Pipeline modular avançado para delimitação de divisões de pastagem (piquetes) e talhões cadastrais:
- **Mosaicos Temporais:** Sentinel-2 Harmonized CS+ (2025)
- **IA Segment Anything:** SAMGeo (Segment Anything Model - ViT-H / ViT-L / ViT-B)
- **Regularização Cadastral:** Grafo planar, retificação de cercas mourão a mourão e eliminação de ilhas
- **Segmentação Pura (Sem Filtro de Pastagem):** Extração de **todos os piquetes/talhões físicos** dentro do perímetro do CAR
- **Filtro Zonal MapBiomas:** Classificação com MapBiomas 10m (Col 4) / 30m (Col 11) até 2025
- **Benchmark de Hardware Automático:** Monitoramento em segundo plano de VRAM, RAM, CPU %, GPU Load %, temperaturas e tempo por etapa, com emissão de relatório técnico auditável.

### 1. Inicialização do Ambiente e Earth Engine
> Se a sua conta Google Cloud exigir um ID de projeto específico, informe-o em `ee.Initialize(project='seu-projeto')`.
> O pipeline detecta automaticamente a sessão ativa do Earth Engine.

In [1]:
import os
import sys
import ee
import geopandas as gpd

ee_project ='mapbiomas-brazil'

# Adiciona o diretório ao path para importar o pacote local
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Inicializa o Earth Engine
try:
    ee.Initialize(project=ee_project)
    print('✅ Earth Engine inicializado com sucesso!')
except Exception as e:
    # Exemplo: ee.Initialize(project='ee-vieiramesquita')
    print(f'Aviso na inicialização automática: {e}')
    print('👉 Se necessário, informe o projeto: ee.Initialize(project="...")')

# Importa as funções principais do pacote LAPIG
from lapig_parcel_segmentation import (
    run_analysis_lapig,
    run_batch_analysis_lapig,
    resolve_input_roi,
    HardwareBenchmarkMonitor
)
print('✅ Pacote LAPIG carregado com sucesso!')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


✅ Earth Engine inicializado com sucesso!
✅ Pacote LAPIG carregado com sucesso!


### 2. Inspecionar o Arquivo Vetorial do CAR (Goiás)
Carrega o GeoPackage com os imóveis rurais e lista os primeiros IDs (`cod_imovel`).

In [2]:
input_gpkg = "car_goias_area_imovel_epsg4674_hottest.gpkg"
id_col = "cod_imovel"

gdf_base = gpd.read_file(input_gpkg)
print(f"📋 Total de imóveis encontrados: {len(gdf_base)}")

# Exibe colunas principais dos imóveis
cols = [c for c in [id_col, 'municipio', 'num_area'] if c in gdf_base.columns]
gdf_base[cols].head(10)

📋 Total de imóveis encontrados: 19


,cod_imovel,municipio,num_area
0,GO-5203500-366A293867A24EF5A446DA91E23F8548,Bom Jesus de Goias,4476.3120
1,GO-5203807-146172AFEDAF44FC82CEEC3A2B92D5F1,Britania,2629.7952
2,GO-5204409-01D1AD1F9586415E9D46BF93115D27DF,Caiaponia,798.2250
3,GO-5205521-DA104FC8E15A44DFB6CB4F0656C1E409,Colinas do Sul,896.5830
4,GO-5206404-499D0D75A1394A6D915AE91FF8DB8700,Crixas,233.8072
5,GO-5207402-1D40C8AAD6504E05B30DA04D7E19457F,Edeia,530.0171
6,GO-5208707-4045057E90C948D999280F125FD7FD6B,Goiania,70.2766
7,GO-5208905-F496428818874B9CBBA57AD8132348B1,Goias,10828.8002
8,GO-5208905-08E150868E2A43CDBB23AC8FEE459017,Goias,828.9465
9,GO-5209903-C8653C1E63894890B5883B8529A3A100,Iaciara,532.2021


### 3. Opção A: Executar para um Imóvel Específico (Com Segmentação Pura & Benchmark)

Nesta execução, o pipeline:
1. **Monitora o Hardware:** Mede CPU, RAM, VRAM e GPU em tempo real.
2. **Gera a Segmentação Pura (`piquetes_segmentacao_pura_{ID}.shp`):** Todos os talhões físicos dentro do perímetro do CAR, com colunas de auditoria (`pasto_cov`, `ndvi_med`, `is_pasto`, `classe`).
3. **Gera os Piquetes de Pastagem (`piquetes_finais_cercas_retas_{ID}.shp`):** Apenas os talhões validados pela dominância zonal do MapBiomas.
4. **Gera o Relatório de Benchmark:** `relatorio_benchmark_hardware.txt` e `.json` na pasta do imóvel.

In [15]:
# Seleciona o primeiro imóvel como exemplo
cod_alvo = gdf_base[id_col].iloc[18]
print(f"🎯 Imóvel selecionado: {cod_alvo}")

# Executa a análise retornando tanto a camada de pasto quanto a segmentação pura
gdf_pasto, shp_pasto, gdf_puro, shp_puro = run_analysis_lapig(
    input_target=input_gpkg,
    id_column=id_col,
    id_value=cod_alvo,
    year=2025,                  # Ano da análise (Sentinel-2 e MapBiomas 2025)
    mapbiomas_collection='10m', # Coleção 4 (10m) até 2025
    min_area_ha=2.0,            # Elimina fragmentos menores que 2 ha
    straight_tol=22.0,          # Tolerância de retificação de cerca (m)
    min_pasture_coverage=0.6,  # Dominância zonal de pasto (>= 35%)
    skip_sam_if_exists=True,    # Reutiliza masks.tif se já gerado
    return_pure=True,           # Retorna também o GeoDataFrame e path da segmentação pura
    enable_benchmark=True,       # Ativa o monitoramento contínuo de hardware
    ee_project=ee_project
)

print(f"\n💾 Shapefile Pastagem:        {shp_pasto} ({len(gdf_pasto)} piquetes)")
print(f"💾 Shapefile Segmentação Pura: {shp_puro} ({len(gdf_puro)} talhões)")

IndexError: single positional indexer is out-of-bounds

### 4. Relatório Técnico de Benchmark de Hardware
Exibe as métricas consolidadas de consumo de hardware registradas durante o processamento do imóvel:

In [16]:
report_txt = os.path.join(os.path.dirname(shp_pasto), "relatorio_benchmark_hardware.txt")
if os.path.exists(report_txt):
    with open(report_txt, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("Relatório de benchmark não encontrado.")

📊 RELATÓRIO DE BENCHMARK DE HARDWARE - LAPIG CADASTRAL PIPELINE
⏱️  TEMPO TOTAL DE PROCESSAMENTO: 0:07:40 (460.9s)
🖥️  DISPOSITIVO DE PROCESSAMENTO: NVIDIA GeForce RTX 2080 Ti
--------------------------------------------------------------------------------
⏱️  DISTRIBUIÇÃO DE TEMPO POR ETAPA:
  • Inicialização                              0:00:00 (  0.0%)
  • 1. Resolução da Área de Interesse (ROI)    0:00:01 (  0.3%)
  • 2. Download Mosaico Sentinel-2 (GEE)       0:00:59 ( 12.9%)
  • 3. Inferência SAMGeo (ViT-H GPU)           0:06:32 ( 85.2%)
  • 4. Vetorização e Regularização Cadastral (CPU) 0:00:01 (  0.3%)
  • 5. Segmentação Pura no Perímetro CAR       0:00:00 (  0.0%)
  • 6. Classificação Zonal MapBiomas (GEE/CPU) 0:00:06 (  1.4%)
  • 7. Exportação e Estatísticas Cadastrais    0:00:00 (  0.0%)
--------------------------------------------------------------------------------
🎮 PLACA DE VÍDEO (GPU - NVIDIA GeForce RTX 2080 Ti):
  • Pico de VRAM Alocada:           6.47 GB / 11.00 GB (

### 5. Opção B: Executar em Lote (Batch) para Múltiplos Imóveis
> Itera sequencialmente sobre todos os imóveis do arquivo.
> Gera as pastas individuais `results/{cod_imovel}/`, shapefiles unificados e relatório de benchmark consolidado.

In [3]:
import os
import sys
import ee
import geopandas as gpd

input_gpkg = "car_goias_area_imovel_epsg4674_hottest.gpkg"
id_col = "cod_imovel"

gdf_base = gpd.read_file(input_gpkg)
print(f"📋 Total de imóveis encontrados: {len(gdf_base)}")

# Exibe colunas principais dos imóveis
cols = [c for c in [id_col, 'municipio', 'num_area'] if c in gdf_base.columns]
gdf_base[cols].head(10)
# Seleciona o primeiro imóvel como exemplo
cod_alvo = gdf_base[id_col].iloc[5]
print(f"🎯 Imóvel selecionado: {cod_alvo}")


📋 Total de imóveis encontrados: 19
🎯 Imóvel selecionado: GO-5207402-1D40C8AAD6504E05B30DA04D7E19457F


##### Defina max_imoveis=3 para teste rápido ou max_imoveis=None para a base inteira
gdfs_sucesso, shp_consolidado = run_batch_analysis_lapig(
    input_target=input_gpkg,
    id_column=id_col,
    #id_value=cod_alvo,
    max_imoveis=None,           # None = Todos os imóveis do arquivo
    year=2025,
    mapbiomas_collection='10m',
    min_area_ha=2.0,
    straight_tol=22.0,
    min_pasture_coverage=0.6,
    skip_sam_if_exists=True,
    consolidate_shp=True,
    enable_benchmark=True,
    ee_project=ee_project
)

print(f"\n🏁 Lote concluído!")
print(f"💾 Shapefile Consolidado Pasto: {shp_consolidado}")
print(f"💾 Shapefile Consolidado Puro:  results/piquetes_todos_imoveis_segmentacao_pura.shp")

### 6. Visualização Interativa no JupyterLab (Leaflet / Folium)
Compare a **Segmentação Pura** (todos os talhões físicos) e os **Piquetes de Pasto** validados pelo MapBiomas:

In [ ]:
if 'gdf_puro' in locals() and len(gdf_puro) > 0:
    # Visualiza os talhões da segmentação pura coloridos pela classificação (Pasto vs Outros)
    display(gdf_puro.explore(
        column='classe',
        cmap='tab10',
        tooltip=['id_piquete', 'area_ha', 'classe', 'pasto_cov', 'ndvi_med'],
        popup=True,
        tiles='CartoDB positron',
        legend=True
    ))
elif 'gdf_pasto' in locals() and len(gdf_pasto) > 0:
    display(gdf_pasto.explore(
        column='area_ha',
        cmap='YlGn',
        tooltip=['id_piquete', 'area_ha', 'pasto_cov'],
        popup=True,
        tiles='CartoDB positron'
    ))

In [5]:
input_gpkg = "/mnt/d/samGeoPasture/Teste_Vectorization/CAR_BRASIL_AREA_IMOVEL_EPSG4674.parquet"
id_col = "cod_imovel"

# Defina max_imoveis=3 para teste rápido ou max_imoveis=None para a base inteira
gdfs_sucesso, shp_consolidado = run_batch_analysis_lapig(
    input_target=input_gpkg,
    id_value='GO-5211008-46C08C6235D74D36BE07B5384A17B156', 
    #id_value='MS-5006309-8DC93A48C2424FD7B3EDF235F8B531E1',
    #id_value='GO-5214838-32EB159D2A3C43728A560A168E62D325',
    #max_imoveis=None,           # None = Todos os imóveis do arquivo
    year=2025,
    mapbiomas_collection='10m',
    min_area_ha=2.0,
    straight_tol=22.0,
    min_pasture_coverage=0.6,
    skip_sam_if_exists=True,
    consolidate_shp=True,
    enable_benchmark=True,
    ee_project=ee_project
)

print(f"\n🏁 Lote concluído!")
print(f"💾 Shapefile Consolidado Pasto: {shp_consolidado}")
print(f"💾 Shapefile Consolidado Puro:  results/piquetes_todos_imoveis_segmentacao_pura.shp")


🚜 INICIANDO PROCESSAMENTO EM LOTE (BATCH) - LAPIG
   Arquivo de Entrada: CAR_BRASIL_AREA_IMOVEL_EPSG4674.parquet
   Coluna de ID:       cod_imovel
📋 Total de imóveis a serem processados: 1

################################################################################
🚜 [1/1] PROCESSANDO IMÓVEL: GO-5211008-46C08C6235D74D36BE07B5384A17B156
################################################################################
🧹 [VRAM Clean] Memória GPU liberada: Alocada=9.1 MB | Reservada=14.0 MB
📥 Entrada identificada: Arquivo Vetorial Local (.parquet) -> CAR_BRASIL_AREA_IMOVEL_EPSG4674.parquet
🎯 Imóvel filtrado pela coluna 'cod_imovel' == 'GO-5211008-46C08C6235D74D36BE07B5384A17B156'

🚀 PIPELINE DE SEGMENTAÇÃO CADASTRAL LAPIG
   Imóvel / ID:      GO-5211008-46C08C6235D74D36BE07B5384A17B156
   Pasta de Saída:   results/GO-5211008-46C08C6235D74D36BE07B5384A17B156
   Ano Base:         2025 (Janela seca: meses 03 a 10)
   MapBiomas:        Coleção 10m (cobertura até 2025)
🛰️ Exportando mosaic